In [69]:
from gc import collect

import findspark
import pyspark
from pyspark.sql import SparkSession

In [70]:
"""
Exercise #39 - Critical dates analysis
"""

'\nExercise #39 - Critical dates analysis\n'

In [71]:
findspark.init()
sc = pyspark.SparkContext.getOrCreate()
spark = SparkSession.builder.getOrCreate()

In [72]:
"""
RDD SOLUTION
"""

'\nRDD SOLUTION\n'

In [73]:
inputRDD = sc.textFile("data/sensors.txt")
inputRDD.collect()


['s1,2016-01-01,20.5',
 's2,2016-01-01,30.1',
 's1,2016-01-02,60.2',
 's2,2016-01-02,20.4',
 's1,2016-01-03,55.5',
 's2,2016-01-03,52.5',
 's3,2016-01-03,12.5']

In [74]:
filteredRDD = inputRDD.filter(lambda x: float(x.split(",")[2]) > 50).map(lambda x: (x.split(",")[0], x.split(",")[1]))
filteredRDD.collect()

[('s1', '2016-01-02'), ('s1', '2016-01-03'), ('s2', '2016-01-03')]

In [75]:
mappedRDD = filteredRDD.map(lambda x: x[0])
mappedRDD.collect()

['s1', 's1', 's2']

In [76]:
emptySensorsRDD = inputRDD.map(lambda x: x.split(",")[0]).subtract(mappedRDD)
emptySensorsRDD.collect()

['s3']

In [77]:
emptySensorsRDD = emptySensorsRDD.map(lambda x: (x, []))
emptySensorsRDD.collect()

[('s3', [])]

In [78]:
finalRDD = filteredRDD.groupByKey().map(lambda x: (x[0], list(x[1]))).union(emptySensorsRDD)
finalRDD.collect()

[('s1', ['2016-01-02', '2016-01-03']), ('s2', ['2016-01-03']), ('s3', [])]

In [79]:
"""
SPARKSQL SOLUTION
"""

'\nSPARKSQL SOLUTION\n'

In [80]:
from pyspark.sql.types import ArrayType, StringType

In [81]:
df = spark.read.load("data/sensors.txt", format="csv", sep=",", inferSchema=True, header=False)
df.show()
df.createOrReplaceTempView("sensors")

+---+----------+----+
|_c0|       _c1| _c2|
+---+----------+----+
| s1|2016-01-01|20.5|
| s2|2016-01-01|30.1|
| s1|2016-01-02|60.2|
| s2|2016-01-02|20.4|
| s1|2016-01-03|55.5|
| s2|2016-01-03|52.5|
| s3|2016-01-03|12.5|
+---+----------+----+



In [82]:
def collect_dates(sID, date, PM):
    if PM > 50:
        return [date]
    else:
        return []
spark.udf.register("groupDate", collect_dates, ArrayType(StringType()))

<function __main__.collect_dates(sID, date, PM)>

In [83]:
result = spark.sql("SELECT _c0, groupDate(_c1) FROM sensors "
                   "GROUP BY _c0, _c1, _c2")
result.show()

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "C:\Users\marca\anaconda3\envs\Spark\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 1247, in main
    process()
  File "C:\Users\marca\anaconda3\envs\Spark\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 1239, in process
    serializer.dump_stream(out_iter, outfile)
  File "C:\Users\marca\anaconda3\envs\Spark\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\serializers.py", line 225, in dump_stream
    self.serializer.dump_stream(self._batched(iterator), stream)
  File "C:\Users\marca\anaconda3\envs\Spark\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\serializers.py", line 146, in dump_stream
    for obj in iterator:
  File "C:\Users\marca\anaconda3\envs\Spark\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\serializers.py", line 214, in _batched
    for item in iter